# Agent | Tools

**1. 도구 (Tool)**

https://docs.langchain.com/oss/python/integrations/tools/index#tools-and-toolkits

> "LLM이 사용할 수 있는 구체적인 기술이나 장비"

LLM은 기본적으로 학습된 데이터 내에서만 답변할 수 있으며, 실시간 정보나 정확한 수학 계산에는 취약하다. **Tool**은 이러한 한계를 보완하기 위해 LLM에게 쥐여주는 외부 기능이다.

* **역활:** 외부 API 호출, 웹 검색, 코드 실행, 파일 시스템 접근 등 LLM이 직접 할 수 없는 작업을 대신 수행한다.
* **예시:**
* `Google Search`: 최신 정보를 검색한다.
* `Calculator`: 정확한 수학 계산을 수행한다.
* `Python REPL`: 파이썬 코드를 작성하고 실행한다.



**2. 에이전트 (Agent)**

> "도구를 언제, 어떻게 사용할지 결정하는 두뇌"

**Agent**는 LLM을 추론 엔진(Reasoning Engine)으로 사용하여 사용자의 요청을 해결하기 위한 계획을 세우고 실행하는 주체이다. 단순히 정해진 코드를 순서대로 실행하는 것이 아니라, 상황에 따라 유연하게 행동을 결정한다.

* **역활:** 사용자의 질문을 분석하고, 어떤 **Tool**이 필요한지 판단(Thought)하고, 해당 도구를 실행(Action)한 뒤, 그 결과(Observation)를 보고 다음 행동을 결정하거나 최종 답변을 내놓는다.
* **작동 방식 (ReAct 패턴 예시):**
1. **질문:** "현재 서울 날씨에 맞는 옷차림 추천해줘."
2. **생각(Thought):** "서울의 현재 날씨를 먼저 알아야 한다." -> `Search` 도구 선택
3. **행동(Action):** `Search("서울 현재 날씨")` 실행
4. **관찰(Observation):** "서울 기온 5도, 맑음"이라는 결과 획득
5. **생각(Thought):** "5도면 코트나 패딩이 필요하다."
6. **최종 답변:** "현재 서울은 5도이므로 코트나 가벼운 패딩을 추천합니다."

**Agent와 Tools의 상호작용**

1. **Agent가 입력을 받음**: 사용자의 요청을 LLM으로 분석.
2. **적합한 Tool 선택**: 요청을 처리하는 데 가장 적합한 Tool을 선택.
3. **Tool 실행 및 결과 반환**: Tool을 실행하고 결과를 받아 사용자에게 응답.

In [1]:
# %pip install -Uqqq langchain_openai langchain_community langchain_tavily langgraph wikipedia numexpr 'arxiv<4' ddgs

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')


# Tools

In [3]:
import importlib, pkgutil # 모듈 동적 로드 / 패키지 탐색 유틸

package = importlib.import_module('langchain_community.tools')

# 해당 패키지 경로 아래의 하위 모듈들을 하나씩 순회
for module in pkgutil.iter_modules(package.__path__):
    print(module.name) # 각 모듈(도구) 이름 출력

ainetwork
amadeus
arxiv
asknews
audio
azure_ai_services
azure_cognitive_services
bearly
bing_search
brave_search
cassandra_database
clickup
cogniswitch
connery
convert_to_openai
databricks
dataforseo_api_search
dataherald
ddg_search
e2b_data_analysis
edenai
eleven_labs
few_shot
file_management
financial_datasets
github
gitlab
gmail
golden_query
google_books
google_cloud
google_finance
google_jobs
google_lens
google_scholar
google_serper
google_trends
graphql
human
ifttt
interaction
jina_search
jira
json
memorize
merriam_webster
metaphor_search
mojeek_search
multion
nasa
nuclia
office365
openai_dalle_image_generation
openapi
openweathermap
passio_nutrition_ai
playwright
plugin
polygon
powerbi
pubmed
render
requests
riza
scenexplain
searchapi
searx_search
semanticscholar
shell
slack
sleep
spark_sql
sql_database
stackexchange
steam
steamship_image_generation
tavily_search
vectorstore
wikidata
wikipedia
wolfram_alpha
yahoo_finance_news
you
youtube
zapier
zenguard


### Wikipedia Tool

In [4]:
from langchain_community.tools import WikipediaQueryRun       # 위키피디아 질문 실행 Tool
from langchain_community.utilities import WikipediaAPIWrapper # 위키피디아 검색/요약 API 래퍼 클래스

# 위키피디아 API래퍼를 Tool에 연결
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

print(wiki_tool.run('physical AI')) # 위키피디아 검색/요약 결과

Page: Physical artificial intelligence
Summary: Physical artificial intelligence or physical AI refers to artificial intelligence (AI) systems that perceive, reason about and act within the physical world. These systems generally combine AI models with sensors, control systems, actuators and physical machines such as robots or autonomous vehicles. Physical AI overlaps with embodied artificial intelligence, robotics and autonomous systems, but it emphasizes the complete process of perceiving an environment, motion planning an action and physically executing the task to perform work. This differs from digital AI or generative AI (GenAI), which primarily stays in the information or digital realm.
The term became increasingly prominent during the AI boom in the 2020s as AI development expanded from primarily digital applications toward humanoid robots, self-driving vehicles, smart factories and other autonomous machines. Its boundaries are not standardized, and it is often treated as a con

In [5]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from pprint import pprint

messages = [('human', '걸그룹 튜이드의 멤버 알려줘')]

llm = init_chat_model('gpt-5.4-mini')
# print(llm.invoke('걸그룹 튜이드의 멤버 알려줘')) # 최신 정보 알지 못함

agent = create_agent(
    model = llm,
    tools = [wiki_tool]
)

response = agent.invoke({'messages': messages})
pprint(response)

{'messages': [HumanMessage(content='걸그룹 튜이드의 멤버 알려줘', additional_kwargs={}, response_metadata={}, id='ff0b0d7e-5534-46e2-8fe9-9ef08a0a0f79'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 174, 'total_tokens': 194, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeVgYPzhEpqovl1VJwCWVDF9hmjk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b6-cda8-72b1-a7cf-6a6f3be51a66-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Tweed girl group members'}, 'id': 'call_FF3caYoKqSJgK82F

In [6]:
print(response['messages'][-1].content)

혹시 **“튜이드”**가 아니라 **“T-ara”, “Twice”, “ITZY”** 같은 다른 걸그룹을 말씀하신 걸까요?

제가 바로 확인한 범위에서는 **“튜이드”라는 이름의 걸그룹 정보가 명확하지 않아서 멤버를 확답하기 어렵습니다.**  
원하시면 아래 중 하나로 다시 알려주세요:

- 정확한 그룹 이름
- 영어 표기
- 멤버 이름 일부
- 데뷔 연도 / 소속사

알려주시면 바로 멤버를 정리해드릴게요.


### load_tools

**load_tools 사용가능 목록**

라이브러리를 통해 제공되는 외부 tool들을 langchain-community에서 통합하여 사용할 수 있다.
웬만한 기능들은 langchain 생태계 내에서 쉽게 쓸 수 있다.

https://docs.langchain.com/oss/python/integrations/providers/overview

https://docs.langchain.com/oss/python/integrations/tools

| 도구 이름        | 기능 예시               |
|------------------|------------------------|
| llm-math         | LLM 기반 수학 계산     |
| wikipedia        | 위키백과 검색          |
| serpapi          | 구글 검색 API          |
| requests_get     | HTTP GET 요청          |
| requests_post    | HTTP POST 요청         |
| arxiv            | arXiv 논문 검색        |
| pubmed           | PubMed 논문 검색       |
| dalle            | DALL-E 이미지 생성     |
| bing_search      | Bing 검색              |
| duckduckgo_search| DuckDuckGo 검색        |

### arxiv

In [7]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-5.4-mini')
tools = load_tools(['arxiv','wikipedia'])
agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt = "당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해주세요."
)

messages = [('human', '(2608.26070) 이 논문의 내용을 간단하게 설명해줄래? (한글답변)')]
response = agent.invoke({'messages': messages})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='(2608.26070) 이 논문의 내용을 간단하게 설명해줄래? (한글답변)', additional_kwargs={}, response_metadata={}, id='6098f829-31ff-46d1-94f3-4bb746afc59c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 299, 'total_tokens': 320, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeVqSXWKdbOApt1ZNYG3Tbl9AkRk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b6-f4e0-7543-8dab-1ba6e05c81ec-0', tool_calls=[{'name': 'arxiv', 'args': {'query': '2608.26070'}, 'id': 'call_OeXVJwUS

In [8]:
import requests
import xml.etree.ElementTree as ET
from langchain_core.tools import tool

@tool
def search_arxiv(arxiv_id: str) -> str:
    """ arXiv 논문 ID로 제목, 저자, 초록을 조회합니다. """

    url = "https://export.arxiv.org/api/query"
    response = requests.get(
        url,
        params= {
            "id_list": arxiv_id,
            "max_results": 1
        },
        timeout = 10
    )

    response.raise_for_status()

    root = ET.fromstring(response.text)

    ns = {"atom": "http://www.w3.org/2005/Atom"}
    entry = root.find("atom:entry", ns)

    if entry is None:
        return "논문 정보를 찾을 수 없습니다."

    title = entry.findtext("atom:title", namespaces=ns).strip()
    summary = entry.findtext("atom:summary", namespaces=ns).strip()
    authors = [
        author.findtext("atom:name", namespaces=ns)
        for author in entry.findall("atom:author",ns)
    ]

    return f"""
제목: {title}
저자: {', '.join(authors)}
초록: {summary}
"""


In [9]:
tools = [search_arxiv, wiki_tool]

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요."
)

messages = [('human' '(2608.26070) 이 논문의 내용을 간단하게 설명해줄래? (한글답변)')]
response = agent.invoke({'messages': messages})
print(response['messages'][-1].content)

이 논문 **“Prefix Sliding for efficient test-time scaling”**은,  
**긴 추론을 할 때 메모리와 계산비용을 크게 줄이면서도 성능을 유지하는 방법**을 제안합니다.

### 핵심 아이디어
일반적인 LLM은 추론할 때 지금까지의 모든 토큰을 계속 기억하면서 **전체 attention**을 유지합니다.  
그런데 논문은 **추론이 길어질수록 중간에 나온 많은 토큰은 점점 덜 중요해진다**는 점에 주목합니다.

그래서 **Prefix Sliding**이라는 방식을 사용합니다:
- **prefix(앞부분)**: 시스템 지시문, 도구 사용법처럼 계속 중요한 정보는 유지
- **최근 토큰들**: 현재 진행 중인 추론 내용은 유지
- **오래된 중간 토큰들**: 중요도가 낮아지면 과감히 버림

### 왜 좋은가?
이 방식은 긴 추론을 하더라도 **메모리 사용량을 거의 일정하게 유지**할 수 있어서,
- 더 긴 reasoning이 가능하고
- 속도도 빨라지고
- 비용도 줄어듭니다

### 결과
- **학습 없이도** 기존 모델을 **최대 3배 빠르게** 만들 수 있었고
- **강화학습과 함께 학습**하면 **10만 토큰이 넘는 긴 추론**에서도 더 좋은 성능을 보였다고 합니다.
- 단순히 중간 토큰을 요약하거나, 일반적인 sliding window를 쓰는 것보다 더 효과적이었다고 보고합니다.

### 한 줄 요약
**긴 추론 중 중요하지 않은 과거 토큰을 버려서, LLM의 테스트 시점 계산을 더 효율적으로 만드는 방법**을 제안한 논문입니다.

원하시면 제가 이 논문을 **그림처럼 쉽게 비유해서 설명**해드릴 수도 있어요.


### llm-math

In [10]:
from langchain_community.agent_toolkits.load_tools import load_tools

llm = init_chat_model('gpt-4.1-mini')
# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm필요)
tools = load_tools(['wikipedia', 'llm-math'], llm=llm)

agent = create_agent(
    model = llm,
    tools = tools,
    system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요. 단, 숫자계산은 llm-math 도구를 사용해서 답변에 활용해야 합니다."
)

response = agent.invoke({'messages': '3.5의 3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘.'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='3.5의 3제곱은 몇이야? 그리고 그 결과에 5를 곱해줘.', additional_kwargs={}, response_metadata={}, id='e69684dd-0810-4d12-be1a-d1714226d86a'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 187, 'total_tokens': 208, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_ab8fa114f2', 'id': 'chatcmpl-EHeW0wWgy9upCJ1jm9WYm7UpPzp8u', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b7-1f86-7c83-af3f-edaeacbfe7a5-0', tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '3.5 ** 3'}, 'id': 'call_dL

### duckduckgo
https://docs.langchain.com/oss/python/integrations/tools/ddg

DuckDuckGo는 개인정보 추적 없이 웹 검색을 제공하는 검색 엔진으로,
LangChain에서는 이를 외부 최신 정보 검색용 Tool로 활용한다.

실시간 웹 검색 가능
→ LLM의 knowledge cutoff 이후 이슈(뉴스, 화제, 트렌드)에 대응 가능
로그인/API 키 불필요
→ 실습·교육 환경에서 바로 사용 가능
프라이버시 중심
→ 사용자 검색 이력 추적 없음

LangChain에서 제공하는 DuckDuckGo Tool 차이
- DuckDuckGoSearchRun
    - 검색 결과를 하나의 텍스트 요약으로 반환
    - 빠른 질의응답용에 적합
- DuckDuckGoSearchResults
    - 검색 결과를 리스트(제목, 링크, 스니펫 등 구조화) 형태로 반환
    - 에이전트가 여러 결과를 비교·판단해야 할 때 유리

In [11]:
# 덕덕고 검색 Tool 2종류
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults

ddgs = DuckDuckGoSearchRun() # 검색 결과를 텍스트 요약 형태로 반환
print(ddgs.invoke("Trump's first name?"))  # 문자열 출력
ddgs2 = DuckDuckGoSearchResults() # 검색 결과를 제목/링크/스니펫 형태로 반환

print(ddgs2.invoke("Trump's first name?")) # 결과 출력



Donald Trump - Wikipedia First presidency of Donald Trump - Wikipedia Trump was sworn in as president on January 20, 2017. During his first term, his administration focused on immigration, trade, tax cuts, and reducing government regulations. Trump withdrew the United States from the Trans-Pacific Partnership and announced that the country would leave the Paris Agreement on climate change. [13][14] He supported building a wall along the U.S.-Mexico border and ... If you feel that a man's real last name is whatever last name his ancestors used first, then Trump's real last name might be Drumpf. Donald Trump is the 45th and 47th president of the United States (2017-21; 2025- ). Following his inauguration on January 20, 2025, Trump became only the second president to serve two nonconsecutive terms, the first being Grover Cleveland (1885-89; 1893-97).
snippet: Donald Trump - Wikipedia, title: Donald Trump - Wikipedia, link: https://en.wikipedia.org/wiki/Donald_Trump, snippet: First preside

In [12]:

llm = init_chat_model('gpt-5.4-mini')
# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm필요)
tools = [ddgs2]

agent = create_agent(llm, tools, system_prompt='모르는 정보가 있으면 ddgs tool을 사용해 검색해')

response = agent.invoke({'messages': 'gs25 민음사 빵'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='gs25 민음사 빵', additional_kwargs={}, response_metadata={}, id='fb49840f-8a55-47e5-b2da-c5df9384d21c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 180, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeWCgoypv8Jav9QMB87ddrftZlpz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b7-4ca8-7bd3-8dbf-0454de1ac0ff-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'GS25 민음사 빵'}, 'id': 'call_PLYkreCozkBidpl85REnJ

### tavily-search

https://docs.langchain.com/oss/python/integrations/tools/tavily_search

In [13]:
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(
    max_result= 3,
    topic='general',
    include_images = True,
    search_depth = 'advanced'
)

tavily_tool.invoke('2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?')


{'query': '2026년 8월 현재 대한민국에서 가장 핫한 이슈가 뭐야?',
 'follow_up_questions': None,
 'answer': None,
 'images': ['https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426f67d1b5fa839e454dfe_79_thumbnail2.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426e7fc594ca778cc36ab8_79_2-1.png',
  'https://cdn.prod.website-files.com/66e955c4e382a96d6c5c0548/6a426ed2d1b5fa839e44e068_79_2-2.png',
  'https://lookaside.fbsbx.com/lookaside/crawler/threads/DcNY3WAE9AY/0/image.jpg',
  'https://cdn.wakeupnews.co.kr/news/photo/202601/973_1856_5154.png'],
 'results': [{'url': 'https://www.newsshin.co.kr/news/articleView.html?idxno=',
   'title': "【뉴스신ㅣ2026년 8월 22일(토) ㅣ대한민국 '핫' 이슈】",
   'content': ": 【뉴스신】2026년 8월 22일 '핫' 이슈】 주택 문제와 공급 확대, 청년에게는 진입 장벽이고, 기성세대에게는 자산 문제다.",
   'score': 0.902481,
   'raw_content': None,
   'images': [],
   'id': 'dfb8c2-00'},
  {'url': 'https://highyon.tistory.com/entry/%EB%8F%88-%EB%B2%84%EB%8A%94-%ED%95%AB%EC%9D%B4%EC%8A%88-2026%EB%85%84-8%EC%9B%94-

In [14]:
llm = init_chat_model('gpt-5.4-mini')

tools = [tavily_tool]

agent = create_agent(llm, tools, system_prompt='당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여, 사용자의 질문에 거짓없이 답변을 작성해 주세요.')

response = agent.invoke({'messages': '현재 AI업계에서 가장 핫한 주제가 뭐야?'})

pprint(response)
print("=" * 50)
pprint(response['messages'][-1].content)

{'messages': [HumanMessage(content='현재 AI업계에서 가장 핫한 주제가 뭐야?', additional_kwargs={}, response_metadata={}, id='00f14d47-4763-43c3-988c-e3cef8079ad4'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 1322, 'total_tokens': 1385, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeWGMjQ6wx50u7dXxgSwj0Dqosoq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b7-5ce0-7300-94e1-0e78f5c7b487-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'current hottest topics in AI industry 2026

In [15]:

llm = init_chat_model('gpt-5.4-mini')
# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm필요)
tools = [ddgs2]

agent = create_agent(llm, tools, system_prompt="""
당신은 미국주식시장 분석봇입니다.
사용자가 요청한 기업에 대한 2026년 보고서를 직관적으로 분석해주세요.

# 출력형식
다음 내용을 포함해 표형식 출력 (분석기관별 레코드로 작성)

1. 분석기관명
2. 목표주가범위 (최저 ~ 최대)
3. 전망근거 키워드
4. 신뢰도 지수(1 ~ 10)
""")

response = agent.invoke({'messages': '2026년 애플 주가 전망 분석해 줘'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='2026년 애플 주가 전망 분석해 줘', additional_kwargs={}, response_metadata={}, id='4332b508-cc40-4a70-843f-0c85b9a2058c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 267, 'total_tokens': 305, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeWP47ktZ7cPJeTrKQhys5z7hCsH', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b7-7f93-7a30-9bea-0318ea11e042-0', tool_calls=[{'name': 'duckduckgo_results_json', 'args': {'query': 'Apple 2026 price target analysts 2026

In [16]:
from IPython.display import display, Markdown

display(Markdown(response['messages'][-1].content))

| 분석기관명 | 목표주가범위 (최저 ~ 최대) | 전망근거 키워드 | 신뢰도 지수(1~10) |
|---|---:|---|---:|
| Wedbush (Dan Ives) | $350 ~ $400 | AI 인프라/애플 AI 전환, iPhone 업사이드, 서비스 매출 확대, 2026년 재평가 | 8 |
| Morgan Stanley | $315 ~ $364 | 제품 로드맵 강화, WWDC 기대, AI 기능 고도화, 수익화 가시성 | 8 |
| Citi | $315 ~ $365 | 아이폰 판매 회복, 제품 믹스 개선, 밸류에이션 리레이팅, 안정적 현금흐름 | 7 |
| Rothschild & Co. Redburn | $260 ~ $400 | 폴더블 iPhone 기대, AI 전략, 업그레이드 모멘텀, 실적 상향 가능성 | 6 |

### 한줄 요약
2026년 애플(AAPL)은 **AI 도입 속도, iPhone 교체 수요, 서비스 매출 성장**이 주가의 핵심 변수입니다.  
낙관적 시나리오에선 **$350~$400대**, 보수적 시나리오에선 **$260~$315대**까지도 시장 해석이 갈립니다.

원하시면 다음 단계로  
1) **2026년 애플 주가 상승/하락 시나리오**, 또는  
2) **현재 주가 기준 업사이드/다운사이드 계산**  
까지 이어서 정리해드릴게요.

### @tool

In [17]:
# eval / exec로 문자열 코드 실행
a = 10
print(eval("5 + 3 + a")) # 문자열을 평가해서 결과를 반환
exec("b = 10") # 문자열을 실행 (할당 가능)
print(b)

18
10


In [18]:
from langchain_core.tools import tool

@tool
def simple_calculator(query: str) -> str:
    """산술연산을 위한 간단한 계산기 Tool"""  # 함수 설명(1줄)
    """
    산술연산을 위한 간단한 계산기
    Args:
        query: 계산식
    Return:
        계산식 결과값

    Examples:
    - simple_calculator("5 + 3 - 2") -> "계산 결과: 6"
    - simple_calculator("4 ** 2 / 8") -> "계산 결과: 2"
    """
    try:
        result = eval(query)
        return f"계산 결과 : {result}"
    except Exception as e:
        return f"계산 오류 : {str:e}"
simple_calculator

StructuredTool(name='simple_calculator', description='산술연산을 위한 간단한 계산기 Tool', args_schema=<class 'langchain_core.utils.pydantic.simple_calculator'>, func=<function simple_calculator at 0x0000021F580EF100>)

In [19]:

llm = init_chat_model('gpt-5.4-mini')
# wikipedia, llm-math 도구 로드 (수학도구는 계산용 llm필요)
tools = [simple_calculator]

agent = create_agent(llm, tools, system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요. 단, 숫자계산은 llm-math 도구를 사용해서 답변에 활용해야 합니다.")

response = agent.invoke({'messages': '7 + 3 * 8 이거를 계산해줘'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='7 + 3 * 8 이거를 계산해줘', additional_kwargs={}, response_metadata={}, id='b17936ea-c1ef-42d8-a596-52d82afbfb3e'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 197, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeWZDl1ISSRB8QL0xfMxQ4NXJ3Gv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b7-a721-7de3-a3ea-cf20add86f30-0', tool_calls=[{'name': 'simple_calculator', 'args': {'query': '7 + 3 * 8'}, 'id': 'call_IacKW3f2B6O96ufUvDCM

In [20]:
response = agent.invoke({'messages': '김치볶음밥 레시피?'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='김치볶음밥 레시피?', additional_kwargs={}, response_metadata={}, id='2b2e920d-4e2a-412e-8497-adb8af8f8b2c'),
              AIMessage(content='물론이죠! 간단하고 맛있는 **김치볶음밥 레시피** 알려드릴게요.\n\n## 재료 1~2인분\n- 밥 1공기\n- 김치 1/2컵 정도\n- 김치국물 1~2큰술\n- 다진 마늘 1작은술\n- 대파 조금\n- 식용유 1큰술\n- 참기름 1작은술\n- 간장 1작은술(선택)\n- 설탕 약간(김치가 너무 시면)\n- 계란 1개\n- 김가루, 깨 약간\n\n## 만드는 법\n1. **김치와 대파를 잘게 썰어요.**\n2. 팬에 식용유를 두르고 **대파와 마늘**을 볶아 향을 내요.\n3. **김치**를 넣고 1~2분 정도 충분히 볶아요.\n4. 밥을 넣고 **김치국물**과 함께 잘 섞어가며 볶아요.\n5. 맛을 보고 필요하면 **간장 약간, 설탕 약간**으로 간을 맞춰요.\n6. 마지막에 **참기름** 넣고 한 번 더 볶으면 끝!\n7. 접시에 담고 **계란후라이, 김가루, 깨**를 올리면 더 맛있어요.\n\n## 팁\n- 밥은 **찬밥**이 더 잘 볶아져요.\n- 김치를 먼저 충분히 볶아야 **신맛이 줄고 감칠맛**이 살아나요.\n- 참치, 베이컨, 스팸 넣으면 더 든든해요.\n\n원하시면 제가 **참치김치볶음밥 버전**이나 **스팸 넣는 버전**으로도 바로 알려드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 412, 'prompt_tokens': 195, 'total_tokens': 607, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, '

In [21]:
import json

OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

@tool
def get_current_weather(city="Seoul", units="metric"):
    """
    OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수

    Args:
        - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**
            - 변환예시:
                - 서울 -> Seoul
                - 충남, 충청남도 -> Chungcheongnam-do
                - 부산 -> Busan
        - units: str 온도단위를 설정하는 문자열
          - metric(기본값: 섭씨, 미터)
          - imperial(화씨, 야드)
    Return:
        - str: json 형식으로 변환된 현재 날씨 정보
    """

    url = f'https://api.openweathermap.org/data/2.5/weather?q={city}&appid={OPENWEATHER_API_KEY}&units={units}'
    response = requests.get(url)
    data = response.json()  # json -> dict

    weather_info = {}

    if response.status_code == 200:  # 정상 응답 받은 경우
        weather_description = data['weather'][0]['description']  # 날씨 설명
        temp = data['main']['temp']  # 현재 기온
        temp_feels_like = data['main']['feels_like']  # 체감 온도
        humidity = data['main']['humidity']  # 습도

        weather_info = {
            'city': city,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }

    else:  # 응답 불량
        weather_info = {
            'city': city,
            'description': 'Not Found',
            'temperature': 'Not Found',
            'temperature_feels_like': 'Not Found',
            'humidity': 'Not Found'
        }

    return json.dumps(weather_info)  # dict -> json

get_current_weather

StructuredTool(name='get_current_weather', description='OpenWeather API를 사용해서 사용자가 지정한 도시의 현재 날씨 정보를 가져오는 함수\n\nArgs:\n    - city: str 날씨정보를 가져올 도시 이름. **반드시 영문으로 작성하세요.**\n        - 변환예시:\n            - 서울 -> Seoul\n            - 충남, 충청남도 -> Chungcheongnam-do\n            - 부산 -> Busan\n    - units: str 온도단위를 설정하는 문자열\n      - metric(기본값: 섭씨, 미터)\n      - imperial(화씨, 야드)\nReturn:\n    - str: json 형식으로 변환된 현재 날씨 정보', args_schema=<class 'langchain_core.utils.pydantic.get_current_weather'>, func=<function get_current_weather at 0x0000021F580EDEE0>)

In [22]:
llm = init_chat_model('gpt-5.4-mini')
tools = [simple_calculator,get_current_weather]

agent = create_agent(llm, tools, system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요.")

response = agent.invoke({'messages': '오늘 뭐 입어야 돼? 나 서울 살아.'})
pprint(response)
print("="*10)
print(response['messages'][-1].content)


{'messages': [HumanMessage(content='오늘 뭐 입어야 돼? 나 서울 살아.', additional_kwargs={}, response_metadata={}, id='127e74bb-12ed-4ed2-9a14-6781e999dbde'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 341, 'total_tokens': 364, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeWdbQASeTYL6ZDYikbyQnLDD8kA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b7-b601-7f90-b1d2-cb2faf09e259-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Seoul', 'units': 'metric'}, 'id': 'call_JA

In [23]:
# 한국 기준 현재 날씨/시간을 반환하는 Tool
from datetime import datetime
from pytz import timezone

@tool
def get_current_datetime(format: str='%Y-%m-%d %H:%M:%S') -> str:
    """
    한국기준 현재시각정보를 반환하는 함수
    Args:
        format: 날짜/시각 형식 지정
    Return:
        현재시각 문자열

    get_current_datetime() -> "2026-01-15 12:18:32"
    """

    kst = timezone('Asia/Seoul')
    return datetime.now(kst).strftime(format) # 현재 서울 시간을 받아, format 형식의 문자열을 반환
get_current_datetime

StructuredTool(name='get_current_datetime', description='한국기준 현재시각정보를 반환하는 함수\nArgs:\n    format: 날짜/시각 형식 지정\nReturn:\n    현재시각 문자열\n\nget_current_datetime() -> "2026-01-15 12:18:32"', args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'>, func=<function get_current_datetime at 0x0000021F580EF240>)

In [24]:

@tool
def calculate_age(today_date: str, birth_date: str) -> int:
    """
    오늘날짜, 생년월일을 입력받아 나이를 계산하는 도구
    Args:
        - today_date(str): 오늘 날짜 (yyyy-mm-dd형식)
        - birth_date(str): 생년월일 (yyyy-mm-dd형식)
    Return:
        - 계산된 만나이(int)
    """
    try:
        today = datetime.strptime (today_date, '%Y-%m-%d')
        birthday = datetime.strptime (birth_date, '%Y-%m-%d')

        age = today.year - birthday.year
        if (today.month, today.day) < (birthday.month, birthday.day):
            age -=1
        return age
    except ValueError:
        return "날짜 형식이 올바르지 않습니다. yyyy-mm-dd 형식으로 전달해주세요."

calculate_age.invoke({'today_date': '2026-08-27', 'birth_date':'1920-10-11'})

105

In [25]:
llm = init_chat_model('gpt-5.4-mini')
tools = load_tools(['wikipedia']) + [get_current_datetime,calculate_age]

agent = create_agent(llm, tools, system_prompt="당신은 현명한 챗봇입니다. 주어진 도구를 적절하게 활용하여 답변해주세요.")

response = agent.invoke({'messages': '트럼프 대통령의 현재 나이는?'}, config= {'recursion_limit':10})
pprint(response)
print("="*10)
print(response['messages'][-1].content)


{'messages': [HumanMessage(content='트럼프 대통령의 현재 나이는?', additional_kwargs={}, response_metadata={}, id='f1c05c88-e654-45ad-a183-e02626d6ca56'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 366, 'total_tokens': 419, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EHeWgXhEL5Pq229fKFVeN5pW91Xn6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a045b7-c17b-7d33-b9f5-b6ccad112b65-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'Donald Trump'}, 'id': 'call_EXATmOcyQac1HXZt2ib8CP5j', 

## Memory
agent의 checkpointer속성에 메모리객체를 대화내역을 저장한다.
- 임시저장 InMemorySaver()
- 영구저장 SqliteSaver()

### InMemorySaver

In [26]:
from langgraph.checkpoint.memory import InMemorySaver
llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]
# 체크포인터를 전달하여 대화 상태 저장 가능하도록 에이전트 생성
agent = create_agent(llm, tools, checkpointer=InMemorySaver())

response = agent.invoke(
    input = {'messages': [('human', '안녕! 만나서 반갑다! 나는 min이라고 해. 넌 누구니?')]},
    config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
)

print(response['messages'][-1].content)

안녕 min! 만나서 반가워 😊  
나는 OpenAI가 만든 AI 어시스턴트야. 질문에 답해주고, 글쓰기나 아이디어 정리, 번역, 공부, 코드 같은 걸 도와줄 수 있어.

편하게 뭐든 물어봐!


### sqliteSaver

In [27]:
# Langgraph 상태 저장을 SQLite로 영속화하여 저장하는 체크포인터 패키지
%pip install -Uqqq langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [28]:
from langgraph.checkpoint.sqlite import SqliteSaver
from pprint import pprint

llm = init_chat_model('gpt-5.4-mini')
tools = [TavilySearch()]

with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()

    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human', 'Langchain에 대해 설명해줘.')]},
        config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
    )
    pprint(response)
    print("="*10)
    print(response['messages'][-1].content)
    print("="*10)
    response = agent.invoke(
        input = {'messages': [('human', 'Langgraph에 대해 설명해줘.')]},
        config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
    )
    pprint(response)
    print("="*10)
    print(response['messages'][-1].content)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='6289f995-0991-49b5-9f11-fbdad9061e96'),
              AIMessage(content='LangChain은 **대규모 언어 모델(LLM)을 더 쉽게 활용하기 위한 프레임워크**입니다.  \n한마디로 말하면, **“챗GPT 같은 모델을 그냥 호출하는 수준을 넘어, 실제 앱처럼 연결·조합·자동화할 수 있게 해주는 도구 모음”**이라고 볼 수 있어요.\n\n## 왜 필요한가?\nLLM을 직접 쓰면 보통 이런 일이 필요합니다:\n\n- 프롬프트를 잘 구성해야 함\n- 여러 번 대화한 내용을 기억해야 함\n- 외부 데이터베이스, 문서, API와 연결해야 함\n- 모델 출력 형식을 맞춰야 함\n- 여러 단계를 순서대로 실행해야 함\n\nLangChain은 이런 것들을 **표준화된 방식으로 연결**해 줍니다.\n\n---\n\n## LangChain이 해주는 일\n대표적으로 다음과 같은 기능이 있습니다.\n\n### 1. 프롬프트 관리\n프롬프트 템플릿을 만들어 재사용하기 쉽게 합니다.\n\n### 2. 체인(Chain) 구성\n여러 작업을 순서대로 연결할 수 있습니다.  \n예:\n- 질문 입력\n- 문서 검색\n- 관련 내용 요약\n- 최종 답변 생성\n\n### 3. 외부 데이터 연결\n문서, PDF, 웹페이지, DB, 검색엔진 등을 LLM과 연결할 수 있습니다.\n\n### 4. 메모리 관리\n대화형 앱에서 이전 대화 내용을 기억하게 할 수 있습니다.\n\n### 5. 에이전트(Agent) 기능\nLLM이 상황에 따라 도구를 선택해서 쓰게 할 수 있습니다.  \n예:\n- 계산기\n- 웹 검색\n- 데이터 조회\n- API 호출\n\n---\n\n## 핵심 개념\nLangChain을 이해할 때 자주 나오는 용어들이 있어요.\n\n### 

In [29]:
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    checkpointer.setup()

    agent = create_agent(llm, tools, checkpointer=checkpointer)

    response = agent.invoke(
        input = {'messages': [('human', '오케이 완전 이해했어! 그럼 니가 말해준 langchain, langgraph를 세 줄로 설명해줘')]},
        config = {'configurable': {'thread_id': '100'}} # thread_id로 대화 식별
    )
    pprint(response)
    print("="*10)
    print(response['messages'][-1].content)
    print("="*10)

{'messages': [HumanMessage(content='Langchain에 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='6289f995-0991-49b5-9f11-fbdad9061e96'),
              AIMessage(content='LangChain은 **대규모 언어 모델(LLM)을 더 쉽게 활용하기 위한 프레임워크**입니다.  \n한마디로 말하면, **“챗GPT 같은 모델을 그냥 호출하는 수준을 넘어, 실제 앱처럼 연결·조합·자동화할 수 있게 해주는 도구 모음”**이라고 볼 수 있어요.\n\n## 왜 필요한가?\nLLM을 직접 쓰면 보통 이런 일이 필요합니다:\n\n- 프롬프트를 잘 구성해야 함\n- 여러 번 대화한 내용을 기억해야 함\n- 외부 데이터베이스, 문서, API와 연결해야 함\n- 모델 출력 형식을 맞춰야 함\n- 여러 단계를 순서대로 실행해야 함\n\nLangChain은 이런 것들을 **표준화된 방식으로 연결**해 줍니다.\n\n---\n\n## LangChain이 해주는 일\n대표적으로 다음과 같은 기능이 있습니다.\n\n### 1. 프롬프트 관리\n프롬프트 템플릿을 만들어 재사용하기 쉽게 합니다.\n\n### 2. 체인(Chain) 구성\n여러 작업을 순서대로 연결할 수 있습니다.  \n예:\n- 질문 입력\n- 문서 검색\n- 관련 내용 요약\n- 최종 답변 생성\n\n### 3. 외부 데이터 연결\n문서, PDF, 웹페이지, DB, 검색엔진 등을 LLM과 연결할 수 있습니다.\n\n### 4. 메모리 관리\n대화형 앱에서 이전 대화 내용을 기억하게 할 수 있습니다.\n\n### 5. 에이전트(Agent) 기능\nLLM이 상황에 따라 도구를 선택해서 쓰게 할 수 있습니다.  \n예:\n- 계산기\n- 웹 검색\n- 데이터 조회\n- API 호출\n\n---\n\n## 핵심 개념\nLangChain을 이해할 때 자주 나오는 용어들이 있어요.\n\n### 

In [ ]:
# SQLite 체크포인터(DB)의 특정 thread_id의 대화 메시지 조회
with SqliteSaver.from_conn_string('checkpoint.db') as checkpointer:
    # thread_id가 100인 체크포인트 조회
    checkpointer_tuple = checkpointer.get_tuple({"configurable": {"thread_id": "100"}})

    checkpointer_data = checkpointer_tuple.checkpoint # 원본 데이터 dict
    messages = checkpointer_data['channel_values']['messages'] # 저장된 메시지 목록

    for i,message in enumerate(messages, 1):
        # message.type속성값이 있으면 사용, 없으면 클래스명 사용
        msg_type = getattr(message, 'type', message.__class__.__name__)
        print(f"{i}: [{msg_type}] {message.content}")
        print()
        

1: [human] Langchain에 대해 설명해줘.

2: [ai] LangChain은 **대규모 언어 모델(LLM)을 더 쉽게 활용하기 위한 프레임워크**입니다.  
한마디로 말하면, **“챗GPT 같은 모델을 그냥 호출하는 수준을 넘어, 실제 앱처럼 연결·조합·자동화할 수 있게 해주는 도구 모음”**이라고 볼 수 있어요.

## 왜 필요한가?
LLM을 직접 쓰면 보통 이런 일이 필요합니다:

- 프롬프트를 잘 구성해야 함
- 여러 번 대화한 내용을 기억해야 함
- 외부 데이터베이스, 문서, API와 연결해야 함
- 모델 출력 형식을 맞춰야 함
- 여러 단계를 순서대로 실행해야 함

LangChain은 이런 것들을 **표준화된 방식으로 연결**해 줍니다.

---

## LangChain이 해주는 일
대표적으로 다음과 같은 기능이 있습니다.

### 1. 프롬프트 관리
프롬프트 템플릿을 만들어 재사용하기 쉽게 합니다.

### 2. 체인(Chain) 구성
여러 작업을 순서대로 연결할 수 있습니다.  
예:
- 질문 입력
- 문서 검색
- 관련 내용 요약
- 최종 답변 생성

### 3. 외부 데이터 연결
문서, PDF, 웹페이지, DB, 검색엔진 등을 LLM과 연결할 수 있습니다.

### 4. 메모리 관리
대화형 앱에서 이전 대화 내용을 기억하게 할 수 있습니다.

### 5. 에이전트(Agent) 기능
LLM이 상황에 따라 도구를 선택해서 쓰게 할 수 있습니다.  
예:
- 계산기
- 웹 검색
- 데이터 조회
- API 호출

---

## 핵심 개념
LangChain을 이해할 때 자주 나오는 용어들이 있어요.

### Chain
작업들을 연결한 흐름입니다.

### Prompt Template
프롬프트를 변수화한 템플릿입니다.

### Retriever
관련 문서를 찾아오는 역할입니다.

### Vector Store
문서를 임베딩해서 저장하고 검색하는 저장소입니다.

### Agent
LLM이 어떤 도구를 쓸지 스스로 결정하는 구조입니다.

---

1️⃣ 세션(메모리) 유지 방식
예: store = {}, ChatMessageHistory, InMemorySaver
- 특징
    - 서버 메모리에만 대화 상태를 저장
    - 서버 재시작/재배포 시 모두 사라짐
    - 구현이 가장 단순하고 빠름
- 사용 시기
    - 실습 / 데모 / PoC
    - 단일 서버, 짧은 대화
    - “지금 이 세션에서만 기억하면 되는” 경우
- 장단점
    - ✅ 속도 빠름, 구현 쉬움
    - ❌ 서버 내려가면 기억 소멸
    - ❌ 멀티 서버(스케일아웃) 불가능

2️⃣ SQLite 체크포인터
예: SqliteSaver, checkpoint.db
- 특징
    - 로컬 파일(DB)에 대화 상태 저장
    - 서버 재시작해도 대화 복원 가능
    - 설정/운영 부담이 거의 없음
- 사용 시기
    - 1대 서버 운영
    - “재접속 시 대화 이어가기”가 중요한 서비스
    - 내부 도구, 사내용 챗봇, 파일 기반 서비스
- 장단점
    - ✅ 재시작해도 대화 유지
    - ✅ 설정 간단 (파일 하나)
    - ❌ 동시접속/대량 트래픽에 취약
    - ❌ 운영·분석·확장성 한계

3️⃣ RDB (MySQL / PostgreSQL 등)
실무에서 가장 많이 쓰는 방식

- 특징
    - 대화 내역을 정규화된 테이블로 저장
    - 여러 서버가 공유 DB 사용 가능
    - 사용자/세션/대화/이력 분석까지 가능
- 사용 시기
    - 실서비스(운영 환경)
    - 로그인 사용자 기반 챗봇
    - 고객지원, 상담, 금융, 헬스케어, 교육 서비스
- 장단점
    - ✅ 서버 여러 대에서도 동일한 대화 유지
    - ✅ 로그/분석/감사/리포트 가능
    - ✅ 권한·보안·백업 체계화 가능
    - ❌ 설계/운영 비용 존재

- 요약하자면  
메모리 세션    : 빠르고 간단  
SQLite 같은 파일형 DB : 재접속 기억 + 운영 부담 최소  
RDB    같은 관계형 데이터베이스 : 확장성, 안정성, 분석, 운영  

- 서비스 구상 단계에서  
사용자별 히스토리 관리  
문제 발생 시 감사 로그  
대화 품질/모델 성능 분석  
요약/임베딩/재검색(RAG) 연계  
개인화 서비스(추천, 성향 파악)  
이걸 하려면 RDB 또는 그 이상(이벤트 로그, 데이터 웨어하우스) 가 필요

## Middleware
https://docs.langchain.com/oss/python/langchain/middleware

미들웨어를 통해 에이전트의 추론 과정 중간에 개입하여 내부 동작을 커스터마이징할 수 있다.
Agent를 세부적으로 커스터마이징하기 위한 대부분의 작업을 미들웨어로 할 수 있다.

- 대화 기록 요약
- 동작 중 사용자 입력 대기
- 특정 모델 또는 tool에 대한 호출 제약
- fallback
- PII(개인식별정보) 처리 등

### SummarizationMiddleware

In [31]:
from langchain.agents.middleware import SummarizationMiddleware # 대화 내용 자동 요약

model = init_chat_model('gpt-5.4-mini') # 메인 agent LLM (상대적으로 성능이 좋고 비싼모델)
summary_model = init_chat_model('gpt-5.4-mini') # 요약 model (상대적으로 싼모델)

# 요약 middleware
middleware_summarize = SummarizationMiddleware(
    model = summary_model,
    trigger= ('tokens', 1000), # 누적 1000 tokens 일때 트리거
    keep = ('messages', 1),    # 최근 1개 메시지는 요약 제외
    summary_prompt='다음 대화 내용을 적절하게 요약해주세요. \n{messages}'
)

agent = create_agent(
    model = model,
    tools = [],
    checkpointer= InMemorySaver(),     # 메모리 저장소
    middleware= [middleware_summarize] # 요약 미들웨어 적용
)

In [ ]:
response = agent.invoke(
    input = {'messages': [('human', '뮤지컬 Wicked의 내용을 Elphaba 입장에서 서술해줘 Elphaba역으로 연극에 출연해야되서 준비중이야. 중요한 포인트들을 다 짚어줘!')]},
    config = {'configurable':{'thread_id':'7'}}
)
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='뮤지컬 Wicked의 내용을 Elphaba 입장에서 서술해줘 Elphaba역으로 연극에 출연해야되서 준비중이야. 중요한 포인트들을 다 짚어줘!', additional_kwargs={}, response_metadata={}, id='df27f11b-6a64-4099-b0da-795e25de713b'),
              AIMessage(content='물론이야. **Elphaba의 시선**으로 *Wicked*의 내용을 정리하면서, **무대에서 연기할 때 꼭 잡아야 할 감정선과 핵심 포인트**까지 같이 짚어줄게.  \n연기 준비용으로 쓸 수 있게 **스토리 흐름 + Elphaba의 내면 + 장면별 핵심** 중심으로 정리하겠다.\n\n---\n\n# Wicked를 Elphaba 입장에서 본 이야기\n\n## 1) 나는 왜 처음부터 “다른 존재”였나\n나는 처음부터 사람들에게 환영받는 존재가 아니었어.  \n초록빛 피부를 가지고 태어난 순간부터, 사람들은 나를 한 사람으로 보기보다 **이상한 존재**, **불길한 존재**로 봤지.  \n아버지는 나를 노골적으로 외면했고, 어머니는 내 존재를 부끄러워했어.  \n그래서 나는 어린 시절부터 늘 느꼈어.  \n**“나는 사랑받을 수 없는 사람일까?”**  \n이 감정이 내 모든 행동의 바탕이 돼.\n\n### 연기 포인트\n- Elphaba의 기본 정서는 **분노보다 먼저 상처**야.\n- 겉으로는 강하고 냉소적이지만, 속에는 **“인정받고 싶다”**는 절박함이 있어.\n- 초반의 날카로움은 성격이 아니라 **방어기제**로 이해하면 좋아.\n\n---\n\n## 2) Shiz에서의 시작: 희망과 수치심이 동시에 생긴다\nShiz에 가게 되면서 나는 처음으로 “새로운 삶”을 기대하게 돼.  \n하지만 그곳에서도 나는 쉽게 섞이지 못해.  \n사람들은 내 외모를 보고 놀라고, 나는 그 시선에 익숙해져 있으면서도 여전히 상처받아.\n\n그러다 **Gali

In [34]:
response = agent.invoke(
    input = {'messages': [('human', 'Elphaba가 Glinda를 어떤 심정으로 보는게 좋을까?')]},
    config = {'configurable':{'thread_id':'7'}}
)
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n대화에서는 사용자가 **뮤지컬 *Wicked*를 Elphaba의 시점에서 서술해 달라**고 요청하며, **Elphaba 역으로 연극에 출연하기 위해 준비 중이니 중요한 포인트를 짚어달라**고 했습니다.  \n이에 대해 답변은 **Elphaba의 내면과 스토리 흐름을 중심으로 작품을 정리**하고, **연기할 때 필요한 감정선과 장면별 핵심 포인트**를 자세히 설명했습니다.\n\n핵심 내용은 다음과 같습니다.\n\n- **Elphaba는 초록 피부 때문에 태어날 때부터 소외와 상처를 겪은 인물**이며, 겉의 냉소는 방어기제라는 점\n- **Shiz에서 Galinda(Glinda)와의 관계**를 통해 긴장, 호기심, 우정이 형성되는 과정\n- **Dr. Dillamond를 통해 오즈의 차별과 억압을 깨닫고 정의감이 생기는 흐름**\n- **Fiyero와의 만남으로 사랑과 두려움이 동시에 드러나는 감정선**\n- **Wizard를 만나며 희망이 무너지고 환멸을 겪는 전환점**\n- 결국 세상에 의해 **“Wicked Witch”로 규정되지만, 실제로는 진실과 존엄을 지키려 한 인물**이라는 해석\n- **Glinda와의 관계는 Elphaba의 인간성과 성장, 그리고 마지막까지 남는 연결고리**로 제시됨\n- 마지막에는 **비극 속에서도 선택과 존엄을 지키는 인물**로 정리됨\n\n또한 연기 준비를 위해 Elphaba의 핵심 키워드로\n**상처, 방어, 정의감, 사랑에 대한 두려움, 고독, 존엄**을 제시했고,  \n장면별로는 **초반의 경계심, Glinda와의 긴장, Dillamond 장면의 충격, Fiyero와의 끌림과 두려움, Wizard와의 대면에서의 환멸, 후반부의 비극적 선택**을 강조했습니다.\n\n마지막으로 Elphaba를  \n**“오해받은 마녀가 아니라, 끝까지 진실과 존엄을 포기하지 않은 사람”**

In [35]:
response = agent.invoke(
    input = {'messages': [('human', '또 메소드 연기가 필요한 부분이 있을까?')]},
    config = {'configurable':{'thread_id':'7'}}
)
pprint(response)
print("="*10)
print(response['messages'][-1].content)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n대화는 **뮤지컬 *Wicked*의 Elphaba 관점 해석과 연기 준비**를 중심으로 진행되었습니다.\n\n핵심 요약:\n- 처음에는 사용자가 **Elphaba 시점으로 작품을 정리해 달라**고 요청했고, 답변은 Elphaba의 **상처, 소외, 정의감, 사랑에 대한 두려움, 존엄**을 중심으로 스토리와 감정선을 설명했습니다.\n- 특히 **Glinda와의 관계**, **Dillamond를 통해 드러나는 차별 인식**, **Fiyero와의 감정 변화**, **Wizard와의 대면 이후의 환멸**, 그리고 마지막의 **비극 속 선택과 존엄**이 중요한 포인트로 정리되었습니다.\n- 이후 사용자가 **Elphaba가 Glinda를 어떤 심정으로 보는지** 물었고, 답변은 이를 **질투만이 아닌 복합적인 감정**으로 설명했습니다.\n  - 초반: 경계심, 거리감, 불신\n  - 중반: 질투, 부러움, 짜증과 호기심\n  - 깊어질수록: 이해, 동정, 애정\n  - 후반: 신뢰, 미안함, 유대감\n- 최종적으로 Elphaba에게 Glinda는 **라이벌이면서도 친구, 거울이자 증인**, 그리고 **자신을 오해하면서도 떠나지 않는 특별한 존재**로 정리되었습니다.\n\n한 줄로 말하면,  \n**이 대화는 Elphaba의 내면과 Glinda와의 복합적인 관계를 분석하며, 연기할 때 감정의 결을 어떻게 잡아야 하는지에 초점을 맞춘 내용이었습니다.**', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='c6eef2f3-4c2c-4335-9ce6-3afd1df2c3ba'),
              HumanMessage(content='또 메소드 연기가 필요한 부분이 있을까?', additional_kwargs={}, response

### PIIMiddleware
**PII (Personally Identifiable Information) 개인식별정보 처리**

https://docs.langchain.com/oss/python/langchain/middleware/built-in#pii-detection

In [44]:
from langchain.agents.middleware import PIIMiddleware # PII(개인정보) 탐지/처리 미들웨어

middleware_email = PIIMiddleware(
    pii_type = 'email',
    detector= r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}", # 이메일 정규 표현식 ?@?.? 형식
    strategy= 'redact', # 삭제 처리
    apply_to_input=True # apply_to_input -> 입력값 처리
)

middleware_credit_card = PIIMiddleware(
    pii_type = 'credit_card',
    detector= r'(?:\d{4}[-\s]?){3}\d{4}|\d{4}[-\s]?\d{6}[-\s]?\d{5}', # card 정규 표현식
    strategy= 'mask', # 일부만 남기고 마스킹
    apply_to_input=True
)

middleware_api_key = PIIMiddleware(
    pii_type = 'api_key',
    detector= r"sk-[a-zA-Z0-9-_]{10}", # openai api key 정규 표현식
    strategy= 'mask',
)

agent = create_agent(
    model = init_chat_model('gpt-5.4-mini'),
    tools = [],
    middleware = [middleware_email, middleware_credit_card, middleware_api_key],
)

In [ ]:
response = agent.invoke(
    {'messages':[('human', 'AWS를 사용하는데 요금이 잘못 청구된 것 같아. 그래서 금액 재조정을 요청하는 메일을 작성하려 해. 영어로 메일내용을 작성해줘. 내 이메일은 min@gamil.com이야')]}
)
pprint(response)


{'messages': [HumanMessage(content='AWS를 사용하는데 요금이 잘못 청구된 것 같아. 그래서 금액 재조정을 요청하는 메일을 작성하려 해. 영어로 메일내용을 작성해줘. 내 이메일은 [REDACTED_EMAIL]이야', additional_kwargs={}, response_metadata={}, id='ed1b0ea4-ff0b-4ff8-be6f-b606f45538d6'),
              AIMessage(content='물론이죠. 아래처럼 정중하고 간결하게 작성하시면 됩니다. 이메일 주소는 그대로 넣어두었습니다.\n\n---\n\n**Subject:** Request for Billing Review and Adjustment\n\nDear AWS Billing Support,\n\nI hope you are doing well.\n\nI am writing to request a review of a recent AWS charge that appears to have been billed incorrectly. I would appreciate it if you could investigate the issue and adjust the amount if an error is confirmed.\n\nMy account email is **[REDACTED_EMAIL]**.\n\nPlease let me know if you need any additional information from my side to review this case.\n\nThank you for your time and support. I look forward to your response.\n\nBest regards,  \n[Your Name]\n\n---\n\n원하시면 제가 이어서  \n1. **더 공손한 버전**,  \n2. **더 강하게 환불/정정 요청하는 버전**,  \n3. **AWS Support 티켓에 바로 넣기 좋은 짧은 버

In [42]:
response = agent.invoke(
    {'messages':[('human', '환불 요청 메일 작성해줘 내 신용카드 번호는 1234-1234-1234-0000 이야. 최근에 쿠팡에서 주문한 5만원짜리 피규어 환불처리해')]}
)
pprint(response)


{'messages': [HumanMessage(content='환불 요청 메일 작성해줘 내 신용카드 번호는 ****-****-****-0000 이야. 최근에 쿠팡에서 주문한 5만원짜리 피규어 환불처리해', additional_kwargs={}, response_metadata={}, id='df8b3272-fb19-4b09-9f27-6c330c1b9ecc'),
              AIMessage(content='물론이죠. 다만 **카드번호 같은 민감한 정보는 메일에 적지 않는 것**이 안전합니다. 저는 그 번호를 메일 본문에 넣는 형태로 작성해드릴 수는 없어요. 대신 **주문정보와 환불 사유**만으로 요청하는 문안을 드릴게요.\n\n아래처럼 보내시면 됩니다:\n\n---\n\n**제목:** 주문 상품 환불 요청드립니다\n\n안녕하세요, 쿠팡 고객센터 담당자님.\n\n최근 쿠팡에서 주문한 **피규어 상품(5만원)**에 대해 환불을 요청드립니다.  \n주문 상세 내역 확인 후 환불 처리 부탁드립니다.\n\n- 주문 상품: 피규어\n- 주문 금액: 50,000원\n- 요청 사항: 환불 처리\n- 결제 정보: 카드 결제\n\n필요한 추가 정보가 있으면 안내 부탁드립니다.  \n빠른 처리 부탁드립니다.\n\n감사합니다.  \n[이름]\n\n---\n\n원하시면 제가 이걸 **더 공손한 버전**, **간단한 버전**, 또는 **쿠팡 고객센터 채팅용 문구**로도 바꿔드릴게요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 255, 'prompt_tokens': 52, 'total_tokens': 307, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 

In [45]:
response = agent.invoke(
    {'messages':[('human', '환경변수 설정을 잘못한 것 같아 내 API 키 좀 봐줘 OPENAI_API_KEY = sk-proj-SDSAD')]}
)
pprint(response)


{'messages': [HumanMessage(content='환경변수 설정을 잘못한 것 같아 내 API 키 좀 봐줘 OPENAI_API_KEY = ****DSAD', additional_kwargs={}, response_metadata={}, id='122b8bce-bf83-4f30-bca8-7de48271c545'),
              AIMessage(content='직접 API 키를 확인하거나 복구해드릴 수는 없어요. 다만 **환경변수 설정이 맞는지 안전하게 점검하는 방법**은 도와드릴게요.\n\n우선 지금처럼 키를 메시지에 붙인 건 좋지 않으니, **이미 노출됐을 가능성이 있으면 키는 즉시 재발급(rotate)하는 걸 권장**합니다.\n\n### 1) 환경변수 이름 확인\nOpenAI 키는 보통 아래처럼 설정합니다.\n\n- Linux / macOS:\n```bash\nexport OPENAI_API_KEY="sk-..."\n```\n\n- Windows PowerShell:\n```powershell\n$env:OPENAI_API_KEY="sk-..."\n```\n\n- Windows CMD:\n```cmd\nset OPENAI_API_KEY=sk-...\n```\n\n### 2) 값 읽히는지 확인\n- Linux / macOS:\n```bash\necho $OPENAI_API_KEY\n```\n\n- PowerShell:\n```powershell\necho $env:OPENAI_API_KEY\n```\n\n- CMD:\n```cmd\necho %OPENAI_API_KEY%\n```\n\n### 3) 코드에서 확인\n예를 들어 Python에서는:\n\n```python\nimport os\nprint(os.getenv("OPENAI_API_KEY"))\n```\n\n> 출력이 `None`이면 환경변수가 안 잡힌 것입니다.  \n> 실제 키가 출력되면 설정은 되어 있지만, **로그에 키를 남기지 않도록 주의**하세요.\n\n### 4) 자

## Streaming
*openai모델은 조직인증된 사용자에 한해서 stream기능을 사용할수 있다.*

In [ ]:
model = init_chat_model('gpt-5.4-mini')
agent = create_agent(model)
# 스트리밍 실행 (메시지 단위로 받음)
stream = agent.stream(
    input = {'messages':[('human','역사상 가장 위대한 한국인은 누구인가요? 3명과 점수까지 주세요.')]},
    stream_mode= 'messages'
)

for chunk, metadata in stream: # chunk(메시지 조각)와 metadata(정보)를 순회
    print(chunk.content, end='', flush=True) # 조각을 이어붙이면서 출력


“역사상 가장 위대한 한국인”은 기준에 따라 달라서 **절대적인 정답은 없습니다**.  
다만 **역사적 영향력, 문화적 파급력, 국가/민족에 남긴 유산**을 종합해 보면 보통 아래 3명이 가장 자주 거론됩니다.

## 1위: 세종대왕 — 98점
- **한글 창제**로 한국인의 문자 생활을 완전히 바꿈
- 과학, 농업, 음악, 천문 등 여러 분야를 크게 발전시킴
- 한국 역사상 가장 넓고 깊은 영향력을 남긴 인물로 평가됨

## 2위: 이순신 — 95점
- 임진왜란에서 조선 수군을 이끌고 나라를 지켜낸 상징적 영웅
- **전술, 리더십, 충성심** 면에서 세계적으로도 높은 평가를 받음
- 한국인에게 “국난 극복”의 상징

## 3위: 안중근 — 90점
- 일제강점기 이전 독립운동의 상징적 인물
- 단순한 의거를 넘어, **동양 평화와 민족 자주**의 이상을 남김
- 한국인의 독립 의지를 대표하는 인물로 기억됨

### 참고
만약 기준을 “학문/문명 기여”로 더 좁히면 **세종대왕**이 1위가 거의 확실하고,  
“군사적 업적” 기준이면 **이순신**이 1위로 올라갈 수 있습니다.  
“독립운동/정신적 상징” 기준이면 **안중근**의 평가가 더 높아질 수 있습니다.

원하시면 제가 다음 중 하나로도 정리해드릴게요:
- **객관식 기준별 TOP 10**
- **위인/군사/문화/과학 분야별 1위**
- **현대 한국인까지 포함한 TOP 3**